# Corpus texte — TTS wolof

## Contexte

Premier chantier TTS du projet sur le wolof. Ce notebook constitue le corpus de
texte de test qui servira à évaluer qualitativement plusieurs modèles TTS. Il tourne
en local et produit un fichier figé — `corpus_test_tts_wolof.json` — que le notebook d'évaluation se contente
de charger : tous les modèles sont ainsi testés sur exactement le même input.

Le corpus croise plusieurs registres et difficultés : questions RAG (usage réel du
projet), wolof oral spontané (KALLAAMA), phrases à nombres, code-switching, et wolof
écrit (texte KALLAAMA).

## Objectif

- Obtenir un corpus de texte réutilisable et reproductible pour évaluer les modèles
  TTS, figé dans un JSON versionné (champs `source` / `license` par catégorie).

## Section 0 — Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
import json, re
import pandas as pd

PROJECT_ROOT = Path(r"C:\dev\noo-far-pipeline")

# rag/ est à la racine du repo ; kallaama vit sous src/ — les deux sur le path
sys.path.insert(0, str(PROJECT_ROOT))            # pour rag/
sys.path.insert(0, str(PROJECT_ROOT / "src"))    # pour kallaama

DATA_DIR = PROJECT_ROOT / "data"
CHECKED_DIR = DATA_DIR / "kallaama" / "wolof" / "speech_dataset" / "checked_transcriptions"
WOLOF_TXT = DATA_DIR / "kallaama" / "wolof" / "text_corpora" / "wolof.txt"   # source wolof_ecrit — ajuste si besoin

In [ ]:
from kallaama import load_corpus, extract_segment
from rag.questions_test_wo import QUESTIONS_TEST_WO

df, stats = load_corpus(CHECKED_DIR)

print(len(df), "segments |", len(QUESTIONS_TEST_WO), "questions RAG")
print("colonnes :", df.columns.tolist())
print("wolof.txt présent :", WOLOF_TXT.exists())

## Section 1 — Construire le corpus de test

### 2.1 Questions RAG (usage réel du projet)

In [ ]:
from rag.questions_test_wo import QUESTIONS_TEST_WO

phrases_questions = [item["question"] for item in QUESTIONS_TEST_WO]
print(f"{len(phrases_questions)} questions RAG")

In [ ]:
for i, item in enumerate(QUESTIONS_TEST_WO):
    q = item["question"]
    print(f"{i:2d} | {len(q.split()):2d} mots | {q}")

In [ ]:
# --- Catégorie : lexique_domaine (ex-questions_rag) ---
# Questions RAG comme sonde du lexique métier Ñoo Far. '?' retiré : lecture
# déclarative, cohérent avec le TTS produit qui vocalisera des réponses, pas
# des questions. Source = contenu projet (pas de licence externe).

def cat_lexique_domaine(questions):
    return [q["question"].rstrip(" ?").strip() for q in questions]

phrases_lexique = cat_lexique_domaine(QUESTIONS_TEST_WO)

for p in phrases_lexique:
    print(" |", p)
print(f"\n{len(phrases_lexique)} phrases")

### 2.2 Phrases avec nombres

In [ ]:
chiffres = df.text_clean.str.contains(r"\d", na=False)
mots_wo  = df.text_clean.str.contains(r"ñaar|ñett|ñeent|juróom|fukk|téeméer|junni", na=False)
print("chiffres & usable :", (chiffres & df.usable).sum())
print("chiffres TOUS     :", chiffres.sum())
print("mots_wo & usable  :", (mots_wo & df.usable).sum())

In [ ]:
# --- Catégorie : phrases_nombres (choix manuel, use_gate=False) ---
# Le gate ASR tue 100% des chiffres (vérifié : chiffres & usable = 0), on le retire.
# Bruit structuré (tél, fréquences, nombres nus, "projet 4R") -> pas de sample aveugle,
# on pioche 4 chiffres arabes + 4 nombres verbalisés wolof à la main.

# Panier A — chiffres arabes en contexte (5-12 mots, hors tél/fréq/4R/nus)
c = df[df.text_clean.str.contains(r"\d", na=False)].copy()
c["nmots"] = c.text_clean.str.split().str.len()
bruit = c.text_clean.str.contains(r"\b(?:33|77|70|76|78)\d|\d{4,}|point|\bFM\b|\bRTS\b|4R|num[eé]ro", na=False)
panierA = c[(c.nmots.between(5,12)) & ~bruit].drop_duplicates("text_clean")
print(f"PANIER A — chiffres en contexte : {len(panierA)}")
for i, t in enumerate(panierA.text_clean):
    print(f"A{i:2d} | {t}")

# Panier B — nombres verbalisés wolof (5-15 mots)
MOTS_NOMBRES_WO = r"ñaar|ñett|ñeent|juróom|fukk|téeméer|junni"
b = df[df.text_clean.str.contains(MOTS_NOMBRES_WO, na=False)].copy()
b["nmots"] = b.text_clean.str.split().str.len()
panierB = b[b.nmots.between(5,15)].drop_duplicates("text_clean")
print(f"\nPANIER B — verbalisé wolof : {len(panierB)}")
for i, t in enumerate(panierB.text_clean.head(20)):
    print(f"B{i:2d} | {t}")

In [ ]:
phrases_nombres = [
"le 26 juin la tawoon",
"nee naa la woon 63% yépp ay jigéen lañu",
"ndax 11 yi  11 yooyu yépp 6 yi ay jigéen la",
"15 jours sax boobu dafay daal di commencé génn",
"ñaare bu prix marché bi nekkee 300 en général Fapal 350 lay lay jëndee",
"man bu ma amoon  ñaari tool am cent kilos phosphates",
"muy  ñenti téeméeri kilos phosphates maanaam quatre cent kilos à l' hectare",
"ba yeneen fukki fan ak juróom ba mu mat  ñent fukki fan ak juróom",
]
assert len(phrases_nombres) == 8
for p in phrases_nombres: print(" |", p)

### 2.3 Phrases avec codeswitch

In [ ]:
# --- Catégorie : phrases_codeswitch ---
cs = df[df.usable & df.has_codeswitch].copy()
cs["nmots"] = cs.text_clean.str.split().str.len()
cs = cs[cs.nmots.between(5, 20)]     # phrases dicibles, ni fragment ni monologue
print(f"vivier CS (usable, 5-20 mots) : {len(cs)}")

# split discursif vs technique — pour décider du biais
TECH = r"information|climat|engrais|météo|meteo|assurance|semence|formation|produit|hectare|litre|kilo|vaccin|maladie|laboratoire"
cs_tech = cs[cs.text_clean.str.contains(TECH, na=False)]
print(f"  dont porteuses d'un terme technique FR : {len(cs_tech)}")

print("\n--- sample(8) brut (random_state=4) ---")
for t in cs.sample(8, random_state=4).text_clean:
    print(" |", t)

print("\n--- sample(8) biaisé technique ---")
for t in cs_tech.sample(min(8, len(cs_tech)), random_state=4).text_clean:
    print(" |", t)

In [ ]:
# --- Catégorie : phrases_codeswitch (biais technique) ---
# CS discursif (mais/donc) déjà connu par l'audit -> peu discriminant.
# On tire dans le vivier porteur d'un terme technique FR (259), plus proche
# du produit et plus dur pour le TTS. Pas de filtre "Fra" : terme métier légitime.
TECH = r"information|climat|engrais|météo|meteo|assurance|semence|formation|produit|hectare|litre|kilo|vaccin|maladie|laboratoire"
cs_tech = df[df.usable & df.has_codeswitch & df.text_clean.str.contains(TECH, na=False)].copy()
cs_tech["nmots"] = cs_tech.text_clean.str.split().str.len()
cs_tech = cs_tech[cs_tech.nmots.between(5, 20)]

phrases_codeswitch = cs_tech.sample(8, random_state=4).text_clean.tolist()
assert len(phrases_codeswitch) == 8
for p in phrases_codeswitch:
    print(" |", p)

### 2.4 Web scraping KALLAAMA

In [ ]:
# --- Catégorie : wolof_ecrit ---
# Source : wolof.txt (texte écrit KALLAAMA, SLR151, CC-BY-4.0). Registre écrit,
# opposé à l'oral spontané des autres cat. Pipeline validé par exploration :
# dénormaliser (ponctuation tokenisée ~92%) AVANT longueur, puis 6-25 mots,
# puis exclusion pulaar [ɗƴ], puis sample reproductible.

def denormaliser(t):
    t = t.strip().strip('"').strip()          # guillemets orphelins de bord
    t = re.sub(r'^[,;:.!?]+\s*', '', t)        # ponctuation orpheline de tête (ex-" , ")
    t = re.sub(r'"\s+', '"', t)                # guillemet interne : "  x -> "x
    t = re.sub(r'\s+"', '"', t)                #                     x " -> x"
    t = re.sub(r'\s+([,.;:!?])', r'\1', t)     # " ," -> ","
    t = re.sub(r'([«(])\s+', r'\1', t)         # "( x" -> "(x"
    t = re.sub(r'\s+([»)])', r'\1', t)         # "x )" -> "x)"
    t = re.sub(r'\s{2,}', ' ', t)              # espaces multiples
    return t.strip()

def cat_wolof_ecrit(chemin, n=5, random_state=4):
    import pandas as pd
    lignes = [l.rstrip("\n") for l in open(chemin, encoding="utf-8")]
    s = pd.Series(lignes).map(denormaliser)
    s = s[~s.str.contains(r"[ɗƴ]", na=False)]              # exclusion pulaar
    s = s[~s.str.match(r"^\s*\d")]                          # <-- fragments de nombre (tri)
    s = s[s.str.split().str.len().between(6, 25)]          # longueur APRÈS dénorm
    print(f"vivier wolof_ecrit : {len(s)} phrases propres")
    return s.sample(n, random_state=random_state).tolist()

phrases_wolof_ecrit = cat_wolof_ecrit(WOLOF_TXT)
for p in phrases_wolof_ecrit:
    print(" |", p)

### 2.5 Phrases kallaama (oral spontané)

In [ ]:
# --- Catégorie : phrases_kallaama (oral spontané natif) ---
phrases_kallaama = (
    df[df.usable & (df.duration >= 3) & (df.duration < 8)]
    .sample(5, random_state=4)["text_clean"]
    .tolist()
)
for p in phrases_kallaama:
    print(" |", p)
print(f"\n{len(phrases_kallaama)} phrases")

### 2.6 Assemblage + écriture du corpus figé

In [ ]:
# --- Assemblage + écriture du corpus figé ---
corpus_test_tts = {
    "lexique_domaine":   {"source": "rag/questions_test_wo.py", "license": None,        "phrases": phrases_lexique},
    "phrases_kallaama":  {"source": "KALLAAMA SLR151",          "license": "CC-BY-4.0", "phrases": phrases_kallaama},
    "phrases_nombres":   {"source": "KALLAAMA SLR151",          "license": "CC-BY-4.0", "phrases": phrases_nombres},
    "phrases_codeswitch":{"source": "KALLAAMA SLR151",          "license": "CC-BY-4.0", "phrases": phrases_codeswitch},
    "wolof_ecrit":       {"source": "KALLAAMA SLR151 (wolof.txt)","license": "CC-BY-4.0","phrases": phrases_wolof_ecrit},
}

# garde-fou : toutes les variables présentes et aux bonnes tailles
attendu = {"lexique_domaine":10, "phrases_kallaama":5, "phrases_nombres":8, "phrases_codeswitch":8, "wolof_ecrit":5}
for cat, d in corpus_test_tts.items():
    assert len(d["phrases"]) == attendu[cat], f"{cat}: {len(d['phrases'])} != {attendu[cat]}"
    print(f"{cat:20s} : {len(d['phrases'])} phrases")

import json
out = PROJECT_ROOT /"tts" / "corpus_test_tts_wolof.json"
with open(out, "w", encoding="utf-8") as f:
    json.dump(corpus_test_tts, f, ensure_ascii=False, indent=2)
print(f"\n✓ écrit : {out}  (total {sum(len(d['phrases']) for d in corpus_test_tts.values())} phrases)")